# Phase 4 Benchmark: Sort / Limit / Distinct

> Benchmark for Phase 4 MXFrame operations using Mojo sort_indices and unique_mask kernels.

This notebook benchmarks Sort, Limit, and Distinct on synthetic grouped data.
Comparisons: MXFrame CPU vs Polars vs Pandas.

In [ ]:
#| hide
#| eval: false

import os
import platform
import sys
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
from max import driver as _driver

from mxframe.lazy_frame import LazyFrame, Scan
from mxframe.lazy_expr import col, lit
from mxframe.custom_ops import clear_cache

try:
    import polars as pl
    POLARS_AVAILABLE = True
except ImportError:
    POLARS_AVAILABLE = False
    print('polars not available — skipping Polars baselines')


In [ ]:
#| hide
#| eval: false

def _safe_gpu_count() -> int:
    try:
        return int(_driver.accelerator_count())
    except Exception:
        return 0

def report_runtime_versions() -> None:
    print("Runtime versions")
    print("================")
    print(f"numpy: {np.__version__}")
    print(f"pandas: {pd.__version__}")
    print(f"pyarrow: {pa.__version__}")
    if POLARS_AVAILABLE:
        print(f"polars: {pl.__version__}")
    else:
        print("polars: not installed")

def report_benchmark_context(dataset_rows: int, dataset_rows_gpu: int) -> None:
    print("Benchmark context")
    print("=================")
    print(f"Python: {sys.version.split()[0]}")
    print(f"Platform: {platform.platform()}")
    print(f"CPU cores: {os.cpu_count()}")
    print(f"MAX accelerators: {_safe_gpu_count()}")
    print(f"Polars available: {POLARS_AVAILABLE}")
    print(f"CPU dataset rows: {dataset_rows:,}")
    print(f"GPU dataset rows: {dataset_rows_gpu:,}")

def summarize_relative(rows, baselines=("Pandas", "Polars")):
    stats = {name: s for name, s in rows}
    for baseline in baselines:
        if baseline not in stats:
            continue
        print(f"\nRelative to {baseline} (min-ms ratio, <1 is faster):")
        for name, sample in rows:
            ratio = sample["min"] / stats[baseline]["min"]
            print(f"  {name:<24} {ratio:.2f}x")

def print_takeaway(rows, target="Pandas", label="MXFrame CPU", close=1.15, beat=0.95):
    stats = {name: s for name, s in rows}
    if target not in stats or label not in stats:
        return
    ratio = stats[label]["min"] / stats[target]["min"]
    if ratio <= beat:
        verdict = "beats"
    elif ratio <= close:
        verdict = "is close to"
    else:
        verdict = "lags"
    print(f"\nTakeaway: {label} {verdict} {target} ({ratio:.2f}x).")

## 1) Synthetic data

In [ ]:
#| hide
#| eval: false

def make_data(n: int = 500_000, n_groups: int = 1000, seed: int = 42) -> pa.Table:
    rng = np.random.default_rng(seed)
    groups = np.array([f'g{i:04d}' for i in range(n_groups)], dtype=object)
    cat = groups[rng.integers(0, n_groups, size=n)]
    vals = rng.standard_normal(n).astype(np.float64)
    return pa.table({'group': cat, 'value': vals})

N = 500_000
data = make_data(N)                            # 1000 groups — CPU + Polars
data_gpu = make_data(N, n_groups=50)           # 50 groups — fits GPU 64-group limit
print(f'CPU data: {len(data):,} rows, {len(pc.unique(data.column("group")))} groups')
print(f'GPU data: {len(data_gpu):,} rows, {len(pc.unique(data_gpu.column("group")))} groups')
report_benchmark_context(len(data), len(data_gpu))
report_runtime_versions()


CPU data: 500,000 rows, 1000 groups
GPU data: 500,000 rows, 50 groups
Benchmark context
Python: 3.12.12
Platform: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
CPU cores: 24
MAX accelerators: 2
Polars available: True
CPU dataset rows: 500,000
GPU dataset rows: 500,000
Runtime versions
numpy: 2.4.1
pandas: 3.0.1
pyarrow: 22.0.0
polars: 1.34.0


## 2) Query implementations

In [ ]:
#| hide
#| eval: false

# ── MXFrame ─────────────────────────────────────────────
def mx_sort(tbl, device='cpu'):
    """GroupBy + Agg + Sort by group."""
    return (
        LazyFrame(Scan(tbl))
        .groupby('group')
        .agg(col('value').sum().alias('total'))
        .sort('group')
        .compute(device=device)
    )

def mx_sort_limit(tbl, n=10, device='cpu'):
    """GroupBy + Agg + Sort + Limit."""
    return (
        LazyFrame(Scan(tbl))
        .groupby('group')
        .agg(col('value').sum().alias('total'))
        .sort('group')
        .limit(n)
        .compute(device=device)
    )

def mx_distinct(tbl, device='cpu'):
    """Distinct groups from raw column."""
    return (
        LazyFrame(Scan(tbl))
        .distinct('group')
        .compute(device=device)
    )

# ── Polars ──────────────────────────────────────────────
def pl_sort(tbl):
    if not POLARS_AVAILABLE: return None
    return (
        pl.from_arrow(tbl)
        .group_by('group').agg(pl.col('value').sum().alias('total'))
        .sort('group')
    )

def pl_sort_limit(tbl, n=10):
    if not POLARS_AVAILABLE: return None
    return (
        pl.from_arrow(tbl)
        .group_by('group').agg(pl.col('value').sum().alias('total'))
        .sort('group').head(n)
    )

def pl_distinct(tbl):
    if not POLARS_AVAILABLE: return None
    return pl.from_arrow(tbl).select('group').unique()

# ── Pandas ──────────────────────────────────────────────
def pd_sort(tbl):
    df = tbl.to_pandas()
    return df.groupby('group', as_index=False)['value'].sum().sort_values('group').reset_index(drop=True)

def pd_sort_limit(tbl, n=10):
    return pd_sort(tbl).head(n)

def pd_distinct(tbl):
    return tbl.to_pandas()['group'].drop_duplicates().reset_index(drop=True)

# Quick preview
print('MXFrame Sort preview (first 5):')
print(mx_sort(data).to_pandas().head())

MXFrame Sort preview (first 5):
   group      total
0  g0000 -20.826462
1  g0001  -9.836839
2  g0002 -16.896812
3  g0003 -27.338821
4  g0004  10.359162


## 3) Benchmark

In [ ]:
#| hide
#| eval: false

COLD_RUNS = 3
HOT_RUNS  = 5

def _bench_stats(fn, cold=COLD_RUNS, hot=HOT_RUNS, clear=False):
    """Return cold/hot timing stats as dictionaries."""
    times_c = []
    for _ in range(cold):
        if clear:
            clear_cache()
        t0 = time.perf_counter()
        fn()
        times_c.append((time.perf_counter() - t0) * 1000.0)

    times_h = []
    for _ in range(hot):
        t0 = time.perf_counter()
        fn()
        times_h.append((time.perf_counter() - t0) * 1000.0)

    c = np.array(times_c)
    h = np.array(times_h)
    return (
        {"min": float(c.min()), "avg": float(c.mean()), "median": float(np.median(c))},
        {"min": float(h.min()), "avg": float(h.mean()), "median": float(np.median(h))},
    )

def _print_table(title, rows):
    print(f'\n{title}')
    print('=' * len(title))
    print(f"  {'Engine':<24} {'Cold min':>10} {'Cold med':>10} {'Hot min':>10} {'Hot med':>10}")
    print(f"  {'-'*24} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
    for name, cold, hot in rows:
        print(f"  {name:<24} {cold['min']:>10.1f} {cold['median']:>10.1f} {hot['min']:>10.1f} {hot['median']:>10.1f}")

# ── GPU readiness ────────────────────────────────────────────────────────
# GPU groupby is limited to 64 groups/launch, so GPU rows use data_gpu (50 groups).
# CPU/Polars/Pandas rows use data (1000 groups).
GPU_READY = False
if _driver.accelerator_count() > 0:
    try:
        _ = mx_sort(data_gpu, device='gpu')
        GPU_READY = True
        print(f'GPU available ({_driver.accelerator_count()} device(s))')
        print('Note: GPU benchmarks use 50-group dataset (GPU kernel limit: 64 groups/launch)')
    except Exception as e:
        print(f'GPU not usable: {e}')
else:
    print('No GPU detected — CPU + Polars + Pandas only')

# ── Sort ────────────────────────────────────────────────
rows_sort = []
c, h = _bench_stats(lambda: mx_sort(data), clear=True)
rows_sort.append(('MXFrame CPU', c, h))
c, h = _bench_stats(lambda: pd_sort(data))
rows_sort.append(('Pandas', c, h))
if GPU_READY:
    c, h = _bench_stats(lambda: mx_sort(data_gpu, device='gpu'), clear=True)
    rows_sort.append(('MXFrame GPU', c, h))
if POLARS_AVAILABLE:
    c, h = _bench_stats(lambda: pl_sort(data))
    rows_sort.append(('Polars', c, h))
_print_table('Sort (groupby+agg+sort)', rows_sort)
sort_hot = [(name, hot) for name, _, hot in rows_sort]
summarize_relative(sort_hot)
print_takeaway(sort_hot, target='Pandas', label='MXFrame CPU')
if GPU_READY:
    print_takeaway(sort_hot, target='Pandas', label='MXFrame GPU')

# ── Sort + Limit ────────────────────────────────────────
rows_sl = []
c, h = _bench_stats(lambda: mx_sort_limit(data), clear=True)
rows_sl.append(('MXFrame CPU', c, h))
c, h = _bench_stats(lambda: pd_sort_limit(data))
rows_sl.append(('Pandas', c, h))
if GPU_READY:
    c, h = _bench_stats(lambda: mx_sort_limit(data_gpu, device='gpu'), clear=True)
    rows_sl.append(('MXFrame GPU', c, h))
if POLARS_AVAILABLE:
    c, h = _bench_stats(lambda: pl_sort_limit(data))
    rows_sl.append(('Polars', c, h))
_print_table('Sort + Limit(10)', rows_sl)
sl_hot = [(name, hot) for name, _, hot in rows_sl]
summarize_relative(sl_hot)
print_takeaway(sl_hot, target='Pandas', label='MXFrame CPU')
if GPU_READY:
    print_takeaway(sl_hot, target='Pandas', label='MXFrame GPU')

# ── Distinct — no aggregation so any N works on GPU ──────
rows_dist = []
c, h = _bench_stats(lambda: mx_distinct(data), clear=True)
rows_dist.append(('MXFrame CPU', c, h))
c, h = _bench_stats(lambda: pd_distinct(data))
rows_dist.append(('Pandas', c, h))
if GPU_READY:
    c, h = _bench_stats(lambda: mx_distinct(data, device='gpu'), clear=True)
    rows_dist.append(('MXFrame GPU', c, h))
if POLARS_AVAILABLE:
    c, h = _bench_stats(lambda: pl_distinct(data))
    rows_dist.append(('Polars', c, h))
_print_table('Distinct (unique groups) — GPU uses full 1000g', rows_dist)
dist_hot = [(name, hot) for name, _, hot in rows_dist]
summarize_relative(dist_hot)
print_takeaway(dist_hot, target='Pandas', label='MXFrame CPU')
if GPU_READY:
    print_takeaway(dist_hot, target='Pandas', label='MXFrame GPU')


GPU available (2 device(s))
Note: GPU benchmarks use 50-group dataset (GPU kernel limit: 64 groups/launch)

Sort (groupby+agg+sort)
  Engine                     Cold min   Cold med    Hot min    Hot med
  ------------------------ ---------- ---------- ---------- ----------
  MXFrame CPU                  3152.8     3182.8        4.3        4.5
  Pandas                         13.1       13.3       12.7       12.9
  MXFrame GPU                  6003.6     6062.3        9.0        9.8
  Polars                         19.4       30.9       19.7       20.6

Relative to Pandas (min-ms ratio, <1 is faster):
  MXFrame CPU              0.34x
  Pandas                   1.00x
  MXFrame GPU              0.71x
  Polars                   1.55x

Relative to Polars (min-ms ratio, <1 is faster):
  MXFrame CPU              0.22x
  Pandas                   0.64x
  MXFrame GPU              0.46x
  Polars                   1.00x

Takeaway: MXFrame CPU beats Pandas (0.34x).

Takeaway: MXFrame GPU beats Pand